In [ ]:

# 1. IMPORTING REQUIRED LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Tools for data splitting and preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Machine Learning models (Classification)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


In [ ]:

# 2. LOADING AND EXPLORING THE DATASET

df = pd.read_csv('/kaggle/input/datasets/fedesoriano/heart-failure-prediction/heart.csv') 


In [ ]:
# Display the first 5 rows to understand the structure
print("--- First 5 Rows of the Dataset ---")
print(df.head())

In [ ]:

# Display dataset information (columns, data types, and missing values)
print("\n--- Dataset Summary and Info ---")
print(df.info())

In [ ]:
# Shape
print("Shape:", df.shape)

In [ ]:
df.describe()

In [ ]:
# Duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Missing values
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:

# OUTLIER DETECTION USING IQR METHOD



numerical_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']

print("--- Counting Outliers in Each Numerical Column ---")

# Loop through each numerical column to calculate IQR and find outliers
for col in numerical_cols:
    # Calculate the 1st quartile (25th percentile) and 3rd quartile (75th percentile)
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    # Calculate the Interquartile Range (IQR)
    IQR = Q3 - Q1
    
    # Define the lower and upper bounds for acceptable data values
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Count how many rows fall outside the acceptable bounds
    outliers_count = df[(df[col] < lower_bound) | (df[col] > upper_bound)].shape[0]
    
    # Print the results neatly formatted
    print(f"Column {col:<20} -> Contains: {outliers_count} outliers")

In [ ]:

# OUTLIER HANDLING (CAPPING/WINSORIZATION)


# Loop through each numerical column to apply capping
for col in numerical_cols:
    # Recalculate bounds for each column
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Handle outliers by capping them at the lower and upper bounds (Winsorization)
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

print("="*60)
# Display the descriptive statistics summary to verify the data is clean
print("--- Descriptive Statistics After Cleaning ---")
print(df[numerical_cols].describe())


**3. DATA PREPROCESSING & SPLITTING**


In [ ]:

# Define Features (X) and Target (y)
# 'HeartDisease' is the column we want to predict (1 = diseased, 0 = normal)
X = df.drop(columns=['HeartDisease'])
y = df['HeartDisease']

# Identify numerical and categorical columns automatically
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns

In [ ]:

# Pipeline for numerical features: Fill missing values with median, then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline for categorical features: Fill missing values, then convert text to numbers (One-Hot Encoding)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
# Combine both preprocessors into a single ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Split the data into Training set (80%) and Testing set (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data preprocessing steps defined and dataset split successfully!")

**4. MODEL IMPLEMENTATION & TRAINING**

In [ ]:

# Create a pipeline for Model 1: Logistic Regression
model1 = Pipeline(steps=[('preprocessor', preprocessor),
                        ('classifier', LogisticRegression(random_state=42))])

# Create a pipeline for Model 2: Random Forest
model2 = Pipeline(steps=[('preprocessor', preprocessor),
                        ('classifier', RandomForestClassifier(random_state=42))])

# Train both models on the training dataset
print("Training Logistic Regression...")
model1.fit(X_train, y_train)

print("Training Random Forest...")
model2.fit(X_train, y_train)

print("\nBoth models have been trained successfully!")

In [ ]:
print(y_train)

**5. EVALUATION AND MODEL COMPARISON**

In [ ]:

# Dictionary to hold the models for easy looping
models = {'Logistic Regression': model1, 'Random Forest': model2}

for name, model in models.items():
    # Make predictions using the test dataset (20%)
    y_pred = model.predict(X_test)
    
    print(f"{name} Performance")
    # Calculate and display the overall accuracy
    print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
    
    # Detailed classification report (Precision, Recall, F1-Score)
    print("\nDetailed Classification Report:")
    print(classification_report(y_test, y_pred))
    


In [ ]:
    # Plot and display the Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f'Confusion Matrix - {name}')
    plt.show()

## 6. Final Model Comparison and Conclusion

Based on the evaluation results, we compare the performance of both models:

* **Random Forest Classifier:** Achieved an accuracy score of **87.50%**.
* **Logistic Regression:** Achieved an accuracy score of **85.87%**.

### Conclusion:
The **Random Forest Classifier** is the best-performing model for this dataset. It not only provided a higher overall accuracy but also demonstrated superior performance in terms of **Recall (88% for the positive class)** and **F1-Score (0.89)**. In medical diagnosis, high recall is crucial because it ensures that fewer actual heart disease cases are missed by the model. Therefore, we select **Random Forest** as our final predictive model for heart disease classification.